In [1]:
import pandas as pd
import re
import datetime
import ast
import unicodedata

đọc hết các file cần gộp lại

In [2]:
df_topcv = pd.read_csv("TOPCV_Clean.csv")
df_VNW_ITV = pd.read_csv("VNW_ITV.csv")
df_CV = pd.read_csv("DATA_CViet(Sheet1) (1).csv")


đối tên cột để thực hiện chuẩn hoá

In [3]:
df_B = df_VNW_ITV.rename(columns={
    "title": "title",
    "job_field": "job_field",
    "company": "company",
    "salary_max":"salary_ma",
    "salary_min": "salary_mi",
    "salary_max_vnd": "salary_max",
    "salary_min_vnd": "salary_min",
    "salary_is_negotiable": "salary_negotiable",
    "location_primary": "location_city",
    "skills": "skills",
    "industry": "industry",
    "job_level": "job_level",
    "source": "source"
}
)


In [4]:
df_C = df_CV.rename(columns= {
    "title": "title",
    "job_domain": "job_field",
    "company_name": "company",
    "salary_max": "salary_max",
    "salary_min": "salary_min",
    "is_negotiable": "salary_negotiable",
    "city": "location_city",
    "deadline_date": "deadline",
    "skills_standardized": "skills",
    "industry": "industry",
    "experience_min_years": "experience_min",
    "experience_max_years": "experience_max",
    "experience_level": "job_level",
    "source": "source"
})
df_C.dropna(inplace=True, subset=['title'])

thêm các ID chuẩn cho các ngành

In [5]:
df_C["job_id"] = ["crv_" + str(i).zfill(4) for i in range(1, len(df_C) + 1)]

In [6]:
df_topcv["job_id"] = ["tcv_" + str(i).zfill(4) for i in range(1, len(df_topcv) + 1)]
df_A = df_topcv

In [ ]:
columns_STD = list(set(df_A.columns and set(df_B.columns and df_C.columns)))
print("các cột trùng là:")

In [7]:
common_cols = ['job_field', 'title', 'location_city', 'industry', 'salary_negotiable', 'salary_max', 'job_id', 'company', 'deadline', 'job_level', 'salary_min', 'skills', 'source']
common_cols2 = ['job_field', 'title', 'location_city', 'industry', 'salary_negotiable', 'salary_max', 'job_id', 'company', 'job_level', 'salary_min', 'skills', 'source','posted_date']

In [1]:
def merged(pwd1, pwd2, pwd3):
    df_A = pd.read_csv(pwd1)
    df_B = pd.read_csv(pwd2)
    df_C = pd.read_csv(pwd3)

    # nhặt ra cột trùng
    columns_STD = list(set(df_A.columns and set(df_B.columns and df_C.columns)))
    print("các cột trùng là:")

    df_A = df_A[columns_STD]
    df_B = df_B[columns_STD]
    df_C = df_C[columns_STD]

    df = pd.concat([df_A,df_B,df_C])
    print(df.shape)
    return df

NameError: name 'df_A' is not defined

In [9]:
df = pd.concat([df_A,df_B,df_C])
print(df.shape)

(4759, 14)


In [10]:
source_map = {
    "vnw": "VietnamWorks",
    "itv": "ITviec",
    "crv": "CareerViet",
    "tcv": "TopCV"
}

df["source"] = df["source"].replace(source_map)


In [11]:
for col in df.columns:
    n_missing = df[col].isna().sum()
    if n_missing > 0:
        print(f"Cột {col} có {n_missing} giá trị thiếu")

Cột salary_max có 1209 giá trị thiếu
Cột deadline có 1789 giá trị thiếu
Cột salary_min có 1257 giá trị thiếu
Cột skills có 229 giá trị thiếu
Cột posted_date có 2975 giá trị thiếu


In [12]:
df["salary_max"] = df["salary_max"].fillna(0)
df["salary_min"] = df["salary_min"].fillna(0)

df["skills"] = df["skills"].apply(lambda x: [] if pd.isna(x) or str(x).strip() == "" else x)

In [20]:
for col in df.columns:
        print(f"\nCot: {col}")
        print("-" * 30)
        # Đếm giá trị
        counts = df[col].value_counts()
        # Tạo DataFrame cho export
        result_df = pd.DataFrame({
            'Gia_tri': counts.index,
            'So_luong': counts.values,})
        # In kết quả
        for value, count in counts.items():
            print(f"{value}: {count} lan ")
            
        print(f"Total records: {len(df)}")
        print(f"Null values: {df[col].isnull().sum()}")
        
        # Export CSV
        filename = f"analysis_{col}.csv"
        result_df.to_csv(filename, index=False, encoding='utf-8')
        print(f"Saved: {filename}")


Cot: job_field
------------------------------
Developer: 1343 lan 
Other: 679 lan 
Design: 578 lan 
Unspecified: 304 lan 
Sales: 283 lan 
Tester: 279 lan 
Data: 273 lan 
PM/BA: 252 lan 
AI/ML: 211 lan 
Infrastructure: 156 lan 
DevOps: 134 lan 
Security: 112 lan 
Support: 56 lan 
Business: 31 lan 
Mobile: 14 lan 
Frontend: 11 lan 
Fullstack: 11 lan 
Software: 10 lan 
Backend: 8 lan 
Architecture: 6 lan 
Education: 5 lan 
Cloud: 3 lan 
Total records: 4759
Null values: 0
Saved: analysis_job_field.csv

Cot: title
------------------------------
Thiết kế đồ họa (Graphic Design): 461 lan 
Kinh doanh phần mềm: 231 lan 
Software Engineer: 110 lan 
Công nghệ thông tin khác: 109 lan 
Backend Developer: 97 lan 
Business Analyst (Phân tích nghiệp vụ): 81 lan 
Fullstack Developer: 73 lan 
Product Owner/Product Manager: 57 lan 
IT Helpdesk/IT support: 53 lan 
Mobile Developer: 51 lan 
DevOps Engineer: 48 lan 
business analyst: 47 lan 
IT Project Manager: 47 lan 
Manual Tester: 46 lan 
Frontend Devel

In [14]:



# Hàm chuyển đổi và lọc
def clean_industry(val):
    if pd.isna(val) or val == "[]":
        return []
    try:
        # Chuyển chuỗi thành list (vd: "['CNTT','nghỉ thứ 7']" -> ['CNTT','nghỉ thứ 7'])
        items = ast.literal_eval(val) if isinstance(val, str) else val
        
        # Nếu phần tử là list và chứa dict
        if isinstance(items, list):
            cleaned = []
            for item in items:
                if isinstance(item, dict):
                    name = item.get("name", "").lower()
                    if "nghỉ thứ 7" not in name and "chuyên môn" not in name:
                        cleaned.append(item)
                elif isinstance(item, str):
                    if "nghỉ thứ 7" not in item.lower() and "chuyên môn" not in item.lower():
                        cleaned.append(item)
            return cleaned
        return []
    except Exception:
        return []

# Áp dụng vào cột industry
df["industry"] = df["industry"].apply(clean_industry)



In [15]:
# Chuẩn hóa giá trị
df["job_field"] = df["job_field"].str.lower()
# Map lại
mapping = {
    "developer": "Developer",
    "design": "Design",
    "other": "Other",
    "unspecified": "Unspecified",  # giữ riêng
    "sales": "Sales",
    "ai/ml": "AI/ML",
    "pm/ba": "PM/BA",
    "ba/pm/po": "PM/BA",
    "tester": "Tester",
    "qa/testing": "Tester",
    "data": "Data",
    "infra": "Infrastructure",
    "devops": "DevOps",
    "security": "Security",
    "support": "Support",
    "business": "Business",
    "mobile": "Mobile",
    "fullstack": "Fullstack",
    "frontend": "Frontend",
    "backend": "Backend",
    "software": "Software",
    "architecture": "Architecture",
    "education": "Education",
    "cloud": "Cloud"
}

df["job_field"] = df["job_field"].map(mapping).fillna(df["job_field"])


In [16]:
import pandas as pd
import re
from datetime import datetime
import unicodedata

# ==== 1) Đọc file & đổi tên cột ====

# ==== 2) Chuẩn hoá chuỗi thô ====
def normalize_raw_date(s):
    # None/NaN
    if pd.isna(s):
        return None
    s = str(s)

    # Loại bỏ BOM, normalize unicode, strip space
    s = s.replace("\ufeff", "")
    s = unicodedata.normalize("NFC", s).strip()

    # Nếu là các chuỗi thể hiện null
    if s.lower() in {"nan", "none", "null", ""}:
        return None

    # Bỏ phần giờ nếu có " HH:MM:SS" ở cuối
    s = re.sub(r"\s+\d{2}:\d{2}:\d{2}$", "", s)

    # Thay các dấu phân cách (., -) về /
    # Nhưng chú ý: không đụng tới dạng yyyy-mm-dd (để parse riêng)
    # Cách làm: nếu là dd?mm?yyyy -> chuẩn hoá về dd/mm/yyyy
    if re.fullmatch(r"\d{1,2}[\/\.\-]\d{1,2}[\/\.\-]\d{2,4}", s):
        s = re.sub(r"[\.|-]", "/", s)  # dd-mm-yyyy | dd.mm.yyyy -> dd/mm/yyyy

        # Bổ sung zero dẫn nếu thiếu (1/9/2025 -> 01/09/2025) cho chắc
        m = re.fullmatch(r"(\d{1,2})/(\d{1,2})/(\d{2,4})", s)
        if m:
            d, mth, y = m.groups()
            if len(y) == 2:
                # nếu có năm 2 chữ số -> đoán 20yy (ít gặp)
                y = "20" + y
            s = f"{int(d):02d}/{int(mth):02d}/{int(y):04d}"
        return s

    # Nếu là yyyy/mm/dd -> giữ nguyên dạng này để parse sau
    if re.fullmatch(r"\d{4}[/-]\d{1,2}[/-]\d{1,2}", s):
        # Bổ sung zero dẫn
        m = re.fullmatch(r"(\d{4})[/-](\d{1,2})[/-](\d{1,2})", s)
        if m:
            y, mth, d = m.groups()
            s = f"{int(y):04d}-{int(mth):02d}-{int(d):02d}"
        return s

    return s or None

df["posted_date_raw"] = df["deadline"].apply(normalize_raw_date)

# ==== 3) Thử parse theo nhiều pattern cụ thể trước, rồi mới fallback ====
patterns = [
    "%d/%m/%Y",   # dd/mm/yyyy (sau khi đã normalize)
    "%Y-%m-%d",   # yyyy-mm-dd
    "%Y/%m/%d",   # yyyy/mm/dd (phòng khi còn sót)
    "%d/%m/%y",   # dd/mm/yy hiếm gặp nhưng thêm cho chắc
]

def parse_many(s):
    if s is None:
        return pd.NaT
    # Thử từng pattern "cứng"
    for fmt in patterns:
        try:
            return datetime.strptime(s, fmt)
        except ValueError:
            pass
    # Cuối cùng: bắt-all của pandas (dayfirst để ưu tiên dd/mm/yyyy)
    try:
        return pd.to_datetime(s, errors="raise", dayfirst=True, infer_datetime_format=True)
    except Exception:
        return pd.NaT

df["posted_date_dt"] = df["posted_date_raw"].apply(parse_many)

# ==== 4) Báo cáo các giá trị chưa parse được (để bạn xem xét) ====
bad_mask = df["posted_date_dt"].isna()
bad_values = (
    df.loc[bad_mask, "deadline"]
      .astype(str)
      .value_counts(dropna=False)
      .rename_axis("raw_value")
      .reset_index(name="count")
)
print("Số dòng KHÔNG parse được:", bad_mask.sum())
print("Một vài giá trị gây lỗi (nếu có):")
print(bad_values.head(20))

# ==== 5) Chuẩn format về dd/mm/yyyy ====
df["deadline"] = df["posted_date_dt"].dt.strftime("%d/%m/%Y")

# (Tuỳ chọn) kiểm tra các dòng NaN sau khi chuẩn hoá
still_nan = df[df['deadline'].isna()][["posted_date_raw"]].head(10)
print("Ví dụ các dòng vẫn NaN sau chuẩn hoá:")
print(still_nan)

# ==== 6) Dọn cột phụ & Lưu ====
df = df.drop(columns=["posted_date_raw", "posted_date_dt"])


Số dòng KHÔNG parse được: 1789
Một vài giá trị gây lỗi (nếu có):
  raw_value  count
0       nan   1789
Ví dụ các dòng vẫn NaN sau chuẩn hoá:
  posted_date_raw
0            None
1            None
2            None
3            None
4            None
5            None
6            None
7            None
8            None
9            None


In [17]:
df.shape

(4759, 14)

In [18]:
df["skills"] = df["skills"].astype(str).str.lower()
df["industry"] = df["industry"].apply(lambda x: [s.lower() for s in x])


In [19]:
df.to_csv("Jobs_IT.csv",columns=['job_id', 'title', 'job_field', 'company', 'salary_max', 'salary_min','salary_negotiable', 'location_city', 'posted_date','deadline', 'skills', 'industry', 'job_level', 'source'],index=False, encoding="utf-8-sig")

In [ ]:
def adjust_salary(df):
    df_new = df.copy()
    def fix_value(x):
        if pd.isna(x) or x == 0:
            return None
        if x > 500_000_000:   # quá lớn → chia 1000
            return x / 1000
        elif x < 500_000:     # quá nhỏ → nhân 26
            return x * 26
        else:
            return x
    df_new["Gia_tri"] = df_new["Gia_tri"].apply(fix_value)
    return df_new



In [ ]:

df_min_adjusted = adjust_salary(df)
df_max_adjusted = adjust_salary(df)

print(df_min_adjusted.head(10))
print(df_max_adjusted.head(10))

NameError: name 'df_min' is not defined